# HA#5 — Sarcasm Detection with DistilBERT
**CSC620 Natural Language Processing**  
**Author:** Ryan Alvarado

## Part 2: Fine-Tune DistilBERT for Sarcasm Detection

### Step 1: Install Dependencies

In [ ]:
# Install required packages (run once in Colab)
!pip install transformers datasets scikit-learn torch

### Step 2: Imports

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments

### Step 3: Load Dataset

In [ ]:
# Load the News Headlines Dataset for Sarcasm Detection
# Download from: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection

records = []
with open('Sarcasm_Headlines_Dataset.json', 'r') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(df.head())
print(f"Total samples: {len(df)}")
print(f"Sarcastic: {df['is_sarcastic'].sum()} | Not sarcastic: {(df['is_sarcastic'] == 0).sum()}")

### Step 4: Train/Test Split

In [ ]:
texts = df['headline'].tolist()
labels = df['is_sarcastic'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

### Step 5: Tokenize

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(X_test,  truncation=True, padding=True, max_length=128)

### Step 6: Build PyTorch Dataset

In [ ]:
class SarcasmDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = SarcasmDataset(train_encodings, y_train)
test_dataset  = SarcasmDataset(test_encodings,  y_test)

### Step 7: Load Model and Define Training Arguments

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=50,
)

### Step 8: Define Metrics and Train

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall':    recall_score(labels, preds),
        'f1':        f1_score(labels, preds),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

### Step 9: Evaluate and Save Outputs

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

report = classification_report(y_test, preds, target_names=['Not Sarcastic', 'Sarcastic'])
print(report)

# Save classification report
with open('classification_report.txt', 'w') as f:
    f.write("=== DistilBERT Classification Report ===\n\n")
    f.write(report)
    f.write("\n\n=== Naive Bayes Results (from HA3) ===\n\n")
    f.write("# Paste HA3 results here for comparison\n")

# Save predictions
pred_df = pd.DataFrame({'headline': X_test, 'true_label': y_test, 'predicted_label': preds})
pred_df.to_csv('sarcasm_predictions.csv', index=False)
print("Saved classification_report.txt and sarcasm_predictions.csv")

## Part 3: Reflection and Analysis

---

### A. Model Performance

**In what ways did DistilBERT outperform Naive Bayes on this task? Did you observe any shortcomings of the language model?**

*Write your response here (4–5 sentences).*

---

### B. Practical Considerations

**What are the tradeoffs between using Naive Bayes and a fine-tuned LLM? Which model would you choose for a real-world sarcasm detection system, and why?**

*Write your response here (4–5 sentences).*